# 08 Detection To FoodLens Pipeline

This notebook connects multi-food detection outputs to the existing FoodLens classifier. The detector proposes candidate food regions; each crop is classified with the ResNet50 FT-V2 Food-101 model and then passed through the decision-layer logic.

This is the bridge notebook before adding multi-food outputs to the app.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any

import json
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch import nn
from torchvision import models, transforms

In [ ]:
@dataclass(frozen=True)
class Config:
    DETECTION_DIR: Path = Path("/kaggle/working/results/multi_food_detection")
    ARTIFACT_DIR: Path = Path("/kaggle/input/models/tuannm3823/food101-baseline-artifacts/pytorch/default/1/food101-baseline-artifacts")
    MODEL_PATH: Path = ARTIFACT_DIR / "resnet50_ft_v2_best.pth"
    CLASS_NAMES_PATH: Path = ARTIFACT_DIR / "class_names.json"
    CALIBRATION_PATH: Path = ARTIFACT_DIR / "calibration.json"
    OUTPUT_DIR: Path = Path("/kaggle/working/results/multi_food_foodlens_pipeline")
    IMAGE_SIZE: int = 224
    TOP_K: int = 5
    AUTO_CONFIDENCE: float = 0.70
    SUGGEST_CONFIDENCE: float = 0.35


CFG = Config()
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CFG

In [ ]:
def read_json(path: Path, default: Any) -> Any:
    if not path.exists():
        return default
    return json.loads(path.read_text())


def make_classifier_head(in_features: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, 101),
    )


def load_classifier() -> tuple[nn.Module, list[str], float, torch.device]:
    class_names = read_json(CFG.CLASS_NAMES_PATH, [])
    if len(class_names) != 101:
        raise ValueError("class_names.json must contain 101 Food-101 labels.")

    calibration = read_json(CFG.CALIBRATION_PATH, {})
    temperature = float(calibration.get("temperature", 1.0))
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = models.resnet50(weights=None)
    model.fc = make_classifier_head(model.fc.in_features)
    model.load_state_dict(torch.load(CFG.MODEL_PATH, map_location=device))
    model.to(device)
    model.eval()
    return model, class_names, temperature, device


model, class_names, temperature, device = load_classifier()
temperature, device

In [ ]:
preprocess = transforms.Compose(
    [
        transforms.Resize((CFG.IMAGE_SIZE, CFG.IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)


def classify_crop(crop_path: Path) -> list[tuple[str, float]]:
    image = Image.open(crop_path).convert("RGB")
    image_tensor = preprocess(image).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(image_tensor).cpu()
        probabilities = F.softmax(logits / temperature, dim=1)
        top_probabilities, top_indices = probabilities.topk(CFG.TOP_K, dim=1)
    return [
        (class_names[class_index], float(confidence))
        for class_index, confidence in zip(top_indices[0].tolist(), top_probabilities[0].tolist())
    ]


def decision_band(top_confidence: float) -> str:
    if top_confidence >= CFG.AUTO_CONFIDENCE:
        return "auto_accept"
    if top_confidence >= CFG.SUGGEST_CONFIDENCE:
        return "suggest"
    return "confirm"

In [ ]:
detections_path = CFG.DETECTION_DIR / "detections.csv"
detections_df = pd.read_csv(detections_path)

rows = []
for row in detections_df.itertuples(index=False):
    predictions = classify_crop(Path(row.crop_path))
    top_label, top_confidence = predictions[0]
    output = row._asdict()
    output.update(
        {
            "foodlens_top_label": top_label,
            "foodlens_top_confidence": top_confidence,
            "decision_band": decision_band(top_confidence),
            "top_k_predictions": json.dumps(predictions),
        }
    )
    rows.append(output)

multi_food_df = pd.DataFrame(rows)
multi_food_df.to_csv(CFG.OUTPUT_DIR / "multi_food_predictions.csv", index=False)
multi_food_df.head()

## App Integration Notes

The app should consume `multi_food_predictions.csv` or a matching API response shape later. Each detected region needs a bounding box, crop path, top-k predictions, confidence, and decision band.